# 4. Machine learning forecasts

This notebook forecasts the next 4 weeks of demand for Levi's and The Souled Store.

Each forecast is for one fulfilment location and one item. An item is a SKU in the SKU files, and a category in the category files.

What this notebook does:

1. Load the four processed weekly datasets.
2. Put every dataset on the same Monday week label.
3. Build lag features from earlier weeks only.
4. Compare a Random Forest with a previous-week baseline on a chronological holdout.
5. Forecast 4 future weeks for every location-item series.
6. Check coverage, dates, duplicates, negative values, and whether each history is recent or stale.
7. Score the same 4-week forecast on earlier cutoffs, using only information known at each cutoff.

Run the cells from top to bottom. The forecast cell and the rolling-origin cell each train many small models, so the full run can take several minutes.


## 1. Import libraries


In [ ]:
import pandas as pd
import numpy as np
import sklearn
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)


## 2. Load the four weekly datasets

These files were built in the earlier notebooks. This notebook reads them and does not change the raw data.


In [ ]:
levis_sku = pd.read_csv(
    "../data/processed/levis_Order_data_cleaned_weekly_filled.csv"
)
souledstore_sku = pd.read_csv(
    "../data/processed/thesouledstore_Oder_data_cleaned_weekly_filled.csv"
)
levis_category = pd.read_csv(
    "../data/processed/levis_category_weekly.csv"
)
souledstore_category = pd.read_csv(
    "../data/processed/souledstore_category_weekly.csv"
)

datasets = {
    "Levis SKU": levis_sku,
    "Souled Store SKU": souledstore_sku,
    "Levis Category": levis_category,
    "Souled Store Category": souledstore_category,
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(2))


## 3. Check dates, missing values, and week labels

The SKU files label each week with a Monday. The category files label the same week with the following Tuesday. Demand values are checked here before any week label is moved.


In [ ]:
for name, df in datasets.items():
    df["Week"] = pd.to_datetime(df["Week"], errors="coerce")
    weekdays = sorted(df["Week"].dt.day_name().unique())

    print(f"\n--- {name} ---")
    print("Date range:", df["Week"].min().date(), "to", df["Week"].max().date())
    print("Weekday labels:", weekdays)
    print("Missing values:\n", df.isnull().sum())
    print("Negative demand rows:", int((df["Net Sold Quantity"] < 0).sum()))
    print("Duplicate rows:", int(df.duplicated().sum()))


## 4. Use one Monday label for every dataset

A Tuesday category date is the day after the Monday used in the SKU files. For example, category week `2026-02-24` is the same week as SKU week `2026-02-23`.

The label is moved back by one day. `Net Sold Quantity` is not changed, and no rows are added or removed. Running this cell again is safe: once the weeks are already Mondays, they are left as they are.


In [ ]:
def prepare_weekly(df, item_col):
    """Sort one series at a time and align Tuesday labels to Monday."""
    out = df.copy()
    out["Week"] = pd.to_datetime(out["Week"])
    weekdays = set(out["Week"].dt.day_name().unique())

    if weekdays == {"Tuesday"}:
        out["Week"] = out["Week"] - pd.Timedelta(days=1)
        shifted = True
    elif weekdays == {"Monday"}:
        shifted = False
    else:
        raise ValueError(
            "Expected every week to fall on Monday or Tuesday. "
            f"Found {sorted(weekdays)}."
        )

    out = out.sort_values(["Fulfilment Location", item_col, "Week"])
    key = ["Fulfilment Location", item_col, "Week"]
    if out.duplicated(key).any():
        raise ValueError("Duplicate location-item-week rows after week alignment.")

    if set(out["Week"].dt.day_name().unique()) != {"Monday"}:
        raise ValueError("Week alignment did not produce Monday labels.")

    return out.reset_index(drop=True), shifted


demand_before = {
    "Levis Category": float(levis_category["Net Sold Quantity"].sum()),
    "Souled Store Category": float(souledstore_category["Net Sold Quantity"].sum()),
}

levis_sku, levis_sku_shifted = prepare_weekly(levis_sku, "SKU Code")
souledstore_sku, souled_sku_shifted = prepare_weekly(souledstore_sku, "SKU Code")
levis_category, levis_category_shifted = prepare_weekly(levis_category, "Category")
souledstore_category, souled_category_shifted = prepare_weekly(
    souledstore_category, "Category1"
)

datasets = {
    "Levis SKU": (levis_sku, "SKU Code", levis_sku_shifted),
    "Souled Store SKU": (souledstore_sku, "SKU Code", souled_sku_shifted),
    "Levis Category": (levis_category, "Category", levis_category_shifted),
    "Souled Store Category": (
        souledstore_category,
        "Category1",
        souled_category_shifted,
    ),
}

for name, (df, item_col, shifted) in datasets.items():
    print(f"\n--- {name} ---")
    print("Tuesday labels moved to the previous Monday:", shifted)
    print("Date range:", df["Week"].min().date(), "to", df["Week"].max().date())
    print("Locations:", df["Fulfilment Location"].nunique())
    print("Series:", df.groupby(["Fulfilment Location", item_col]).ngroups)

print("\nCategory demand sum, before -> after")
for name in ["Levis Category", "Souled Store Category"]:
    after = float(datasets[name][0]["Net Sold Quantity"].sum())
    print(f"{name}: {demand_before[name]} -> {after}")
    if demand_before[name] != after:
        raise ValueError("Week alignment changed demand values.")


## 5. Create lag features

The model predicts a week from demand that was already known before that week:

- `lag_1` is the previous week
- `lag_2` is two weeks earlier
- `lag_4` is four weeks earlier
- `rolling_mean_4` is the average of those previous four weeks

Each calculation is done inside one location-item series, after the series is sorted by week. The current week's demand is not included in its own features. Rows that do not yet have four earlier weeks are left out of model training. Those short series are still forecast later, with a separate fallback.


In [ ]:
def create_features(df, item_col):
    df = df.copy()
    df = df.sort_values(["Fulfilment Location", item_col, "Week"])

    group = df.groupby(["Fulfilment Location", item_col])["Net Sold Quantity"]
    df["lag_1"] = group.shift(1)
    df["lag_2"] = group.shift(2)
    df["lag_4"] = group.shift(4)
    df["rolling_mean_4"] = group.transform(
        lambda s: s.shift(1).rolling(4).mean()
    )

    df = df.dropna(subset=["lag_1", "lag_2", "lag_4", "rolling_mean_4"])
    return df


levis_sku_features = create_features(levis_sku, "SKU Code")
souledstore_sku_features = create_features(souledstore_sku, "SKU Code")
levis_category_features = create_features(levis_category, "Category")
souledstore_category_features = create_features(souledstore_category, "Category1")

for name, df in {
    "Levis SKU": levis_sku_features,
    "Souled Store SKU": souledstore_sku_features,
    "Levis Category": levis_category_features,
    "Souled Store Category": souledstore_category_features,
}.items():
    print(f"\n{name}: {df.shape}")
    print(df.head(3))


## 6. Compare Random Forest with a previous-week baseline

The test uses the last 4 weeks of each dataset. Everything earlier is used for training. The split is by date, so future weeks are not mixed into the training period.

The baseline predicts that next week will equal the previous week (`lag_1`).

Two error summaries are saved:

- **MAE** is the average absolute error. A few very large weeks can make this number big.
- **Median AE** is the error of a typical week.

Limitations to keep in mind:

- This is a one-step test. Each test week uses the real previous weeks, so it does not measure how error builds up across a recursive 4-week forecast.
- Levi's category average error is large because South Warehouse has real weeks of several thousand units in the holdout. The median error is the better guide to a typical week.
- Location and item IDs in this comparison are only category codes. They let one shared model see which series a row came from. The future forecasts below use a separate model for each series instead.
- The holdout model is for measuring accuracy. The saved forecasts are fit again on all available history, so the latest weeks are included in the forecast.


In [ ]:
def evaluate_model(df, item_col):
    df = df.copy()
    last_week = df["Week"].max()
    test_start = last_week - pd.Timedelta(weeks=3)

    train = df[df["Week"] < test_start].copy()
    test = df[df["Week"] >= test_start].copy()
    if train.empty or test.empty:
        raise ValueError("Chronological split produced an empty train or test set.")

    location_categories = train["Fulfilment Location"].astype("category")
    item_categories = train[item_col].astype("category")
    train["location_id"] = location_categories.cat.codes
    train["item_id"] = item_categories.cat.codes

    location_map = {
        value: code
        for code, value in enumerate(location_categories.cat.categories)
    }
    item_map = {
        value: code
        for code, value in enumerate(item_categories.cat.categories)
    }
    test["location_id"] = test["Fulfilment Location"].map(location_map).fillna(-1)
    test["item_id"] = test[item_col].map(item_map).fillna(-1)

    features = [
        "location_id",
        "item_id",
        "lag_1",
        "lag_2",
        "lag_4",
        "rolling_mean_4",
    ]
    target = "Net Sold Quantity"

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(train[features], train[target])

    test["ml_prediction"] = model.predict(test[features]).clip(min=0)
    test["baseline_prediction"] = test["lag_1"].clip(lower=0)

    ml_error = (test[target] - test["ml_prediction"]).abs()
    baseline_error = (test[target] - test["baseline_prediction"]).abs()

    return {
        "Dataset": None,
        "Training rows": len(train),
        "Testing rows": len(test),
        "Test start": test["Week"].min().date().isoformat(),
        "Test end": test["Week"].max().date().isoformat(),
        "Random Forest MAE": round(float(ml_error.mean()), 2),
        "Baseline MAE": round(float(baseline_error.mean()), 2),
        "Random Forest median AE": round(float(ml_error.median()), 2),
        "Baseline median AE": round(float(baseline_error.median()), 2),
        "Validation": "Chronological holdout of the last 4 weeks",
    }


results = []
for name, df, item_col in [
    ("Levis SKU", levis_sku_features, "SKU Code"),
    ("Souled Store SKU", souledstore_sku_features, "SKU Code"),
    ("Levis Category", levis_category_features, "Category"),
    ("Souled Store Category", souledstore_category_features, "Category1"),
]:
    result = evaluate_model(df, item_col)
    result["Dataset"] = name
    results.append(result)

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


## 7. Save the model evaluation

The CSV compares Random Forest with the previous-week baseline. The test window is the last 4 weeks, in date order, from 2026-02-02 through 2026-02-23.

Reading the table from the previous cell:

- Levi's SKU: Random Forest average error is 89.10, and the previous-week baseline is 107.02. On a typical week the baseline is closer (median error 5.00 versus 11.26).
- Souled Store SKU: the methods are close. Average error is 3.77 versus 3.87. Median error is 1.23 versus 1.00.
- Levi's category: both errors are large because the holdout contains South Warehouse weeks of several thousand units. Average error is 1,267.91 versus 1,425.84. Median error is 815.64 versus 643.00.
- Souled Store category: Random Forest is lower on both measures (average 9.12 versus 10.90, median 2.40 versus 3.00).

The forest reduces some large misses. For an ordinary week, repeating the previous week is often as close. The saved forecasts therefore use a forest only for a series with at least 20 weeks of its own history.


In [ ]:
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

evaluation_path = output_dir / "ml_model_evaluation.csv"
results_df.to_csv(evaluation_path, index=False)
print("Evaluation results saved.")
print(evaluation_path.resolve())


## 8. Forecast the next 4 weeks

Every location-item combination in the processed data gets four future weeks. The four weeks are the same calendar dates for every series: the next four Mondays after the dataset's last week.

Long series are forecast with their own Random Forest:

- A series needs at least 20 observed weeks. The first 4 weeks only create lags, which leaves 16 weeks for the model to learn from.
- The model sees only that series. A South Warehouse spike cannot be copied onto a store, or onto North Warehouse, that never had those volumes.
- Each leaf averages at least 5 historical weeks, and trees are limited to depth 4. One extreme week therefore cannot be repeated as the entire forecast. This is a limit on how finely the tree can split. It is not a cap on the forecast, and no historical week is deleted.
- The next week's lags are built from history that already happened. Each prediction is then appended and used as an input for the following week.

Short series do not have enough weeks to learn those lags honestly. Their forecast is the average of the available recent demand: the last 4 weeks when those weeks exist, otherwise every week the series has. That same average is used for all 4 future weeks and labeled `recent_average_fallback`. Missing weeks are not filled in with zeros or made-up demand.

Some series end before February 2026. They still receive the common 4 future weeks, using their own last observations. `Last Observed Week` shows when that history stopped.


In [ ]:
MIN_HISTORY_WEEKS = 20
LAG_FEATURES = ["lag_1", "lag_2", "lag_4", "rolling_mean_4"]


def lag_row_from_history(history):
    """Features for the next week, using only the history so far."""
    if len(history) < 4:
        raise ValueError("Lag features need 4 earlier weeks.")
    return {
        "lag_1": history[-1],
        "lag_2": history[-2],
        "lag_4": history[-4],
        "rolling_mean_4": float(np.mean(history[-4:])),
    }


def recent_average(history):
    """Average of the latest available demand, up to 4 weeks."""
    recent = history[-4:] if len(history) >= 4 else history
    return float(np.mean(recent))


def fit_series_model(history):
    rows = []
    targets = []
    for t in range(4, len(history)):
        rows.append(lag_row_from_history(history[:t]))
        targets.append(history[t])

    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=4,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=1,
    )
    model.fit(pd.DataFrame(rows)[LAG_FEATURES], targets)
    return model


def forecast_next_4_weeks(df, item_col):
    df = df.sort_values(["Fulfilment Location", item_col, "Week"]).copy()
    last_week = df["Week"].max()
    future_weeks = pd.date_range(
        last_week + pd.Timedelta(days=7),
        periods=4,
        freq="7D",
    )

    forecasts = []
    grouped = df.groupby(["Fulfilment Location", item_col], sort=True)
    for (location, item), group in grouped:
        group = group.sort_values("Week")
        history = group["Net Sold Quantity"].astype(float).tolist()
        if len(history) == 0:
            continue

        last_observed = group["Week"].max()
        if len(history) >= MIN_HISTORY_WEEKS:
            method = "random_forest"
            model = fit_series_model(history)
            working = history.copy()
            predictions = []
            for _ in future_weeks:
                row = pd.DataFrame([lag_row_from_history(working)])
                prediction = max(0.0, float(model.predict(row[LAG_FEATURES])[0]))
                predictions.append(prediction)
                working.append(prediction)
        else:
            method = "recent_average_fallback"
            predictions = [recent_average(history)] * 4

        for week, prediction in zip(future_weeks, predictions):
            forecasts.append(
                {
                    "Week": week,
                    "Fulfilment Location": location,
                    item_col: item,
                    "Forecast Net Sold Quantity": round(prediction, 2),
                    "Forecast Method": method,
                    "History Weeks": len(history),
                    "Last Observed Week": last_observed,
                }
            )

    forecast = pd.DataFrame(forecasts)
    return forecast.sort_values(
        ["Fulfilment Location", item_col, "Week"]
    ).reset_index(drop=True)


forecast_inputs = [
    ("Levis SKU", levis_sku, "SKU Code"),
    ("Souled Store SKU", souledstore_sku, "SKU Code"),
    ("Levis Category", levis_category, "Category"),
    ("Souled Store Category", souledstore_category, "Category1"),
]

forecast_results = {}
for name, df, item_col in forecast_inputs:
    print(f"Forecasting {name}...")
    forecast = forecast_next_4_weeks(df, item_col)
    forecast_results[name] = forecast
    methods = forecast.groupby("Forecast Method")["History Weeks"].size() / 4
    print(forecast["Week"].dt.date.unique().tolist())
    print("Series by method:")
    print(methods.astype(int).to_string())
    print(f"Rows: {len(forecast)}")


## 9. Save the four forecast files


In [ ]:
file_names = {
    "Levis SKU": "levis_sku_ml_forecast.csv",
    "Souled Store SKU": "souledstore_sku_ml_forecast.csv",
    "Levis Category": "levis_category_ml_forecast.csv",
    "Souled Store Category": "souledstore_category_ml_forecast.csv",
}

for name, forecast in forecast_results.items():
    saved = forecast.copy()
    saved["Week"] = pd.to_datetime(saved["Week"]).dt.strftime("%Y-%m-%d")
    saved["Last Observed Week"] = pd.to_datetime(
        saved["Last Observed Week"]
    ).dt.strftime("%Y-%m-%d")
    path = output_dir / file_names[name]
    saved.to_csv(path, index=False)
    print(f"{name}: saved {len(saved)} rows to {path.resolve()}")


## 10. Validate the saved forecasts

This cell reads the CSVs back from `outputs/` and checks the files that were just written.

Each series must have exactly four future weeks, with no duplicate location-item-week rows and no negative forecasts.

`Last Observed Week` shows the last real week for that series. A history is **recent** when that week is 2026-01-26 or later, which is the four weeks leading up to the dataset's final week of 2026-02-23. A history is **stale** when the last observation is older than that. Stale series still receive a March forecast, but it is based on older demand.


In [ ]:
expected_weeks = ["2026-03-02", "2026-03-09", "2026-03-16", "2026-03-23"]
original_data = {
    "Levis SKU": (levis_sku, "SKU Code"),
    "Souled Store SKU": (souledstore_sku, "SKU Code"),
    "Levis Category": (levis_category, "Category"),
    "Souled Store Category": (souledstore_category, "Category1"),
}

validation_notes = []
freshness_rows = []

for name, filename in file_names.items():
    original, item_col = original_data[name]
    forecast = pd.read_csv(output_dir / filename)
    forecast["Week"] = pd.to_datetime(forecast["Week"])
    forecast["Last Observed Week"] = pd.to_datetime(forecast["Last Observed Week"])

    print(f"\n--- {name} ---")
    keys = ["Week", "Fulfilment Location", item_col]
    combo_counts = forecast.groupby(["Fulfilment Location", item_col]).size()
    original_combos = set(
        map(tuple, original[["Fulfilment Location", item_col]].drop_duplicates().to_numpy())
    )
    forecast_combos = set(
        map(tuple, forecast[["Fulfilment Location", item_col]].drop_duplicates().to_numpy())
    )
    original_locations = set(original["Fulfilment Location"])
    forecast_locations = set(forecast["Fulfilment Location"])
    weeks = sorted(forecast["Week"].dt.strftime("%Y-%m-%d").unique())
    duplicate_keys = int(forecast.duplicated(keys).sum())
    negative = int((forecast["Forecast Net Sold Quantity"] < 0).sum())
    missing = int(forecast["Forecast Net Sold Quantity"].isna().sum())

    history = (
        original.groupby(["Fulfilment Location", item_col])["Net Sold Quantity"]
        .agg(history_max="max", history_median="median")
        .reset_index()
    )
    checked = forecast.merge(history, on=["Fulfilment Location", item_col], how="left")
    above_history = int(
        (checked["Forecast Net Sold Quantity"] > checked["history_max"] + 0.05).sum()
    )

    print("Rows:", len(forecast))
    print("Series:", forecast.groupby(["Fulfilment Location", item_col]).ngroups)
    print("Locations:", forecast["Fulfilment Location"].nunique(),
          "of", len(original_locations))
    print("Weeks:", weeks)
    print("Rows per week:\n", forecast.groupby("Week").size().to_string())
    print("Negative predictions:", negative)
    print("Missing predictions:", missing)
    print("Duplicate location-item-week rows:", duplicate_keys)
    print("Forecasts above that series' historical maximum:", above_history)
    print("Minimum prediction:", forecast["Forecast Net Sold Quantity"].min())
    print("Maximum prediction:", forecast["Forecast Net Sold Quantity"].max())
    print("Methods:")
    print(
        forecast.groupby("Forecast Method")
        .size()
        .div(4)
        .astype(int)
        .rename("series")
        .to_string()
    )
    print("Locations without a forecast:",
          sorted(original_locations - forecast_locations) or "none")

    print("\nLargest predictions:")
    print(
        forecast.nlargest(5, "Forecast Net Sold Quantity")[
            ["Week", "Fulfilment Location", item_col,
             "Forecast Net Sold Quantity", "Forecast Method", "History Weeks"]
        ].to_string(index=False)
    )

    last_data_week = pd.to_datetime(original["Week"]).max()
    recent_start = last_data_week - pd.Timedelta(days=28)
    series_last = forecast.groupby(["Fulfilment Location", item_col])["Last Observed Week"].max()
    on_final_week = int((series_last == last_data_week).sum())
    recent_earlier = int(
        ((series_last >= recent_start) & (series_last < last_data_week)).sum()
    )
    stale = int((series_last < recent_start).sum())
    if on_final_week + recent_earlier + stale != len(series_last):
        problems_fresh = "freshness counts do not add up to the number of series"
    else:
        problems_fresh = None

    print("\nHistory freshness")
    print("Final week in the data:", last_data_week.date())
    print("Recent starts on:", recent_start.date())
    print("Observed on the final week:", on_final_week)
    print("Recent, but not on the final week:", recent_earlier)
    print("Stale histories:", stale)
    freshness_rows.append(
        {
            "Dataset": name,
            "Series": int(len(series_last)),
            "Observed on final week": on_final_week,
            "Recent but not final week": recent_earlier,
            "Recent total": on_final_week + recent_earlier,
            "Stale": stale,
        }
    )

    problems = []
    if problems_fresh:
        problems.append(problems_fresh)
    if weeks != expected_weeks:
        problems.append(f"weeks are {weeks}")
    if not (combo_counts == 4).all():
        problems.append("a series does not have exactly 4 weeks")
    if original_combos != forecast_combos:
        problems.append(
            f"combo coverage mismatch: missing {len(original_combos - forecast_combos)}, "
            f"extra {len(forecast_combos - original_combos)}"
        )
    if original_locations != forecast_locations:
        problems.append("location coverage mismatch")
    if negative or missing or duplicate_keys or above_history:
        problems.append("found negative, missing, duplicate, or out-of-range predictions")
    methods = set(forecast["Forecast Method"])
    if methods - {"random_forest", "recent_average_fallback"}:
        problems.append(f"unexpected methods {methods}")
    rf = forecast["Forecast Method"].eq("random_forest")
    fb = forecast["Forecast Method"].eq("recent_average_fallback")
    if rf.any() and (forecast.loc[rf, "History Weeks"] < MIN_HISTORY_WEEKS).any():
        problems.append("random forest used on a short series")
    if fb.any() and (forecast.loc[fb, "History Weeks"] >= MIN_HISTORY_WEEKS).any():
        problems.append("fallback used on a long series")
    if fb.any():
        flat = forecast.loc[fb].groupby(["Fulfilment Location", item_col])[
            "Forecast Net Sold Quantity"
        ].nunique()
        if (flat != 1).any():
            problems.append("fallback forecast is not flat across the 4 weeks")

    if problems:
        raise AssertionError(f"{name} failed validation: {'; '.join(problems)}")
    print("Validation passed.")

print("\nAll four saved forecast files passed validation.")

freshness_df = pd.DataFrame(freshness_rows)
print("\nRecent versus stale series")
print(
    "Recent: last observed on 2026-01-26 or later. "
    "Stale: last observed before 2026-01-26."
)
print(freshness_df.to_string(index=False))
freshness_path = output_dir / "forecast_history_freshness.csv"
freshness_df.to_csv(freshness_path, index=False)
print("Saved", freshness_path.resolve())


## 11. Why some Levi's category forecasts are still large

The previous forecast file showed North Warehouse jeans moving from about 8 units to 874, then above 1,000. That series' own recent demand is about 1 to 7 units a week, and its largest historical week is 401.

Those jumps came from one shared model trained on every Levi's category together. South Warehouse contains weeks of 6,000 to 11,000 units in January and February 2026, across jeans, polo shirts, shirts, and T-shirts. Store locations stay around 0 to 2 units. The shared model applied a warehouse spike pattern to a much smaller series, and the recursive step then fed that large prediction forward.

The spikes were kept. They sit inside the South Warehouse series on consecutive weeks, so the history is one location's own demand rather than a join that mixed two locations. They may be genuine warehouse volume, or a unit difference that needs a business check. Deleting or capping them would hide that question.

With a separate model, North Warehouse jeans now forecasts about 6.8 units in each of the four weeks. South Warehouse jeans still forecasts roughly 900 to 2,100 because that series itself contains weeks of 1,466 and 6,488. Polo shirts (9 weeks) and shirts (5 weeks) use the recent-average fallback, so their forecasts are the average of the last four actual weeks: about 2,485 and 2,008.

The same spike pattern appears at SKU level. South Warehouse SKU `004JH-0072L` has only four weeks: 3,541, 185, 484, and 61. Its fallback forecast is the average of those weeks, about 1,068, repeated for four weeks. The latest week itself was 61.

This cell prints each Levi's category series beside its own history so a large forecast can be read against the series that produced it.


In [ ]:
category_forecast = pd.read_csv(output_dir / "levis_category_ml_forecast.csv")
detail_rows = []

for (location, category), group in levis_category.groupby(
    ["Fulfilment Location", "Category"]
):
    group = group.sort_values("Week")
    actual = group["Net Sold Quantity"]
    preds = category_forecast[
        (category_forecast["Fulfilment Location"] == location)
        & (category_forecast["Category"] == category)
    ].sort_values("Week")
    detail_rows.append(
        {
            "Fulfilment Location": location,
            "Category": category,
            "History Weeks": len(group),
            "Forecast Method": preds["Forecast Method"].iloc[0],
            "Last 4 actual": [round(v, 2) for v in actual.tail(4).tolist()],
            "Median actual": round(float(actual.median()), 2),
            "Max actual": round(float(actual.max()), 2),
            "Forecasts": preds["Forecast Net Sold Quantity"].tolist(),
        }
    )

detail = pd.DataFrame(detail_rows)
detail["Largest forecast"] = detail["Forecasts"].map(max)
detail = detail.sort_values("Largest forecast", ascending=False)
print(detail.drop(columns="Largest forecast").to_string(index=False))

north = detail[
    (detail["Fulfilment Location"] == "North Warehouse")
    & (detail["Category"] == "Jeans")
].iloc[0]
print("\nNorth Warehouse jeans")
print("Last 4 actual weeks:", north["Last 4 actual"])
print("Forecasts:", north["Forecasts"])
if max(north["Forecasts"]) >= 50:
    raise AssertionError(
        "North Warehouse jeans moved far above its recent demand again."
    )
print(
    "These forecasts stay near the recent weekly demand of about 1 to 7 units."
)

print("\nSouth Warehouse series with forecasts above 500")
large = detail[
    detail["Fulfilment Location"].eq("South Warehouse")
    & detail["Forecasts"].map(lambda values: max(values) > 500)
]
print(
    large[
        ["Category", "Forecast Method", "History Weeks",
         "Last 4 actual", "Max actual", "Forecasts"]
    ].to_string(index=False)
)
print(
    "\nSouth Warehouse jeans, polo shirts, and shirts contain the thousand-unit weeks. "
    "Their forecasts are large because that demand is in their own history. "
    "Polo shirts and shirts have fewer than 20 weeks, so they use the recent-average fallback. "
    "North Warehouse and the stores stay at their own small levels."
)


## 12. Four-week rolling-origin evaluation

Section 6 asks an easier question. It holds out the last four weeks, and each test week still uses the real previous week. That is a one-step score.

This section asks the question the March forecasts have to answer: from a cutoff week, how close are the next four weeks?

- There are 8 cutoffs, one week apart, from 2025-12-08 through 2026-01-26. After the last cutoff, the next four observed weeks run through 2026-02-23.
- A series is scored at a cutoff only when that cutoff week was observed and the series already has at least 20 weeks of history. Shorter series use the recent-average fallback, so they are not used to judge the forest.
- The next four weeks must also exist for that series. Actual demand after the cutoff is used only to score the forecast.
- The Random Forest is trained only on weeks up to and including the cutoff. It predicts the next week, then feeds that prediction into the following week.
- The baseline repeats the cutoff week's demand for all four weeks. This is the previous-week baseline from section 6, carried across the horizon.
- Large historical weeks stay in the training data. Nothing is deleted or capped.

The March forecast files and `ml_model_evaluation.csv` are left as they are. This test is an extra backward-looking check.


In [ ]:
N_ROLLING_ORIGINS = 8


def recursive_random_forest_forecast(history):
    """Forecast 4 weeks from one series, using only that history."""
    model = fit_series_model(history)
    working = list(history)
    predictions = []
    for _ in range(4):
        row = pd.DataFrame([lag_row_from_history(working)])
        prediction = max(0.0, float(model.predict(row[LAG_FEATURES])[0]))
        predictions.append(prediction)
        working.append(prediction)
    return predictions


def rolling_origin_scores(df, item_col):
    """Score 4-week forecasts at several past cutoffs."""
    weeks = [pd.Timestamp(week) for week in sorted(df["Week"].unique())]
    origins = weeks[:-4][-N_ROLLING_ORIGINS:]
    scored_rows = []

    for _, group in df.groupby(["Fulfilment Location", item_col], sort=False):
        group = group.sort_values("Week")
        week_list = [pd.Timestamp(week) for week in group["Week"]]
        values = group["Net Sold Quantity"].astype(float).tolist()
        position = {week: index for index, week in enumerate(week_list)}

        for origin in origins:
            pos = position.get(origin)
            if pos is None or pos + 1 < MIN_HISTORY_WEEKS:
                continue
            future_weeks = [
                origin + pd.Timedelta(days=7 * step) for step in (1, 2, 3, 4)
            ]
            if any(week not in position for week in future_weeks):
                continue

            history = values[: pos + 1]
            predictions = recursive_random_forest_forecast(history)
            baseline = history[-1]
            for horizon, (prediction, week) in enumerate(
                zip(predictions, future_weeks), start=1
            ):
                if week <= origin:
                    raise AssertionError("A scored week was not after the cutoff.")
                actual = values[position[week]]
                scored_rows.append(
                    {
                        "horizon": horizon,
                        "rf_error": abs(prediction - actual),
                        "baseline_error": abs(baseline - actual),
                    }
                )

    return origins, pd.DataFrame(scored_rows)


def summarize_scores(dataset_name, origins, scores):
    """Turn scored weeks into one row per horizon, plus an all-horizon row."""
    cutoff_start = origins[0].date().isoformat()
    cutoff_end = origins[-1].date().isoformat()
    pairs = len(scores) // 4
    rows = []

    def one_row(label, part):
        rf_closer = int((part["rf_error"] < part["baseline_error"]).sum())
        baseline_closer = int((part["baseline_error"] < part["rf_error"]).sum())
        ties = int((part["rf_error"] == part["baseline_error"]).sum())
        rf_mae = float(part["rf_error"].mean())
        baseline_mae = float(part["baseline_error"].mean())
        if rf_mae < baseline_mae:
            lower_average = "Random Forest"
        elif baseline_mae < rf_mae:
            lower_average = "Repeat last week"
        else:
            lower_average = "Tie"
        return {
            "Dataset": dataset_name,
            "Horizon": label,
            "Cutoff start": cutoff_start,
            "Cutoff end": cutoff_end,
            "Series-origin pairs": pairs,
            "Scored weeks": len(part),
            "Random Forest MAE": round(rf_mae, 2),
            "Repeat last week MAE": round(baseline_mae, 2),
            "Random Forest median AE": round(float(part["rf_error"].median()), 2),
            "Repeat last week median AE": round(
                float(part["baseline_error"].median()), 2
            ),
            "Random Forest closer": rf_closer,
            "Baseline closer": baseline_closer,
            "Ties": ties,
            "Lower average error": lower_average,
        }

    for horizon in (1, 2, 3, 4):
        rows.append(
            one_row(
                f"{horizon} week ahead",
                scores[scores["horizon"] == horizon],
            )
        )
    rows.append(one_row("All 4 weeks", scores))
    return rows


def explain_rolling_result(dataset_name, table):
    """Describe, in plain language, which method was closer."""
    overall = table[table["Horizon"] == "All 4 weeks"].iloc[0]
    horizons = table[table["Horizon"] != "All 4 weeks"]
    baseline_horizons = horizons.loc[
        horizons["Lower average error"].eq("Repeat last week"), "Horizon"
    ].tolist()
    forest_horizons = horizons.loc[
        horizons["Lower average error"].eq("Random Forest"), "Horizon"
    ].tolist()

    print(f"\n{dataset_name}")
    print(
        f"Scored {int(overall['Series-origin pairs'])} series-cutoff pairs "
        f"from {overall['Cutoff start']} to {overall['Cutoff end']}."
    )
    print(
        "Average error over all 4 weeks: "
        f"Random Forest {overall['Random Forest MAE']}, "
        f"repeat last week {overall['Repeat last week MAE']}."
    )
    print(
        "Typical-week error (median): "
        f"Random Forest {overall['Random Forest median AE']}, "
        f"repeat last week {overall['Repeat last week median AE']}."
    )
    if overall["Lower average error"] == "Random Forest":
        print("Random Forest has the lower average error on this dataset.")
    elif overall["Lower average error"] == "Repeat last week":
        print("Repeating the last week has the lower average error on this dataset.")
    else:
        print("The two methods have the same average error on this dataset.")

    if overall["Repeat last week median AE"] < overall["Random Forest median AE"]:
        print("On a typical week, repeating the last week is closer.")
    elif overall["Random Forest median AE"] < overall["Repeat last week median AE"]:
        print("On a typical week, Random Forest is closer.")
    else:
        print("On a typical week, the two methods are equally close.")

    if baseline_horizons:
        print(
            "Repeating the last week has the lower average error at: "
            + ", ".join(baseline_horizons)
            + "."
        )
    if forest_horizons:
        print(
            "Random Forest has the lower average error at: "
            + ", ".join(forest_horizons)
            + "."
        )


rolling_tables = []
for name, df, item_col in forecast_inputs:
    print(f"Scoring {name}...")
    origins, scores = rolling_origin_scores(df, item_col)
    if scores.empty:
        raise AssertionError(f"{name} produced no rolling-origin scores.")
    if len(origins) != N_ROLLING_ORIGINS:
        raise AssertionError(f"{name} did not use {N_ROLLING_ORIGINS} cutoffs.")
    table = pd.DataFrame(summarize_scores(name, origins, scores))
    rolling_tables.append(table)
    explain_rolling_result(name, table)

rolling_df = pd.concat(rolling_tables, ignore_index=True)
print("\nRolling-origin results")
print(rolling_df.to_string(index=False))

rolling_path = output_dir / "ml_rolling_origin_evaluation.csv"
rolling_df.to_csv(rolling_path, index=False)
print("\nSaved", rolling_path.resolve())
print("The one-step results in ml_model_evaluation.csv were not replaced.")


## 13. What the four-week test says about Random Forest

The saved March forecasts are unchanged. This is the reading of `outputs/ml_rolling_origin_evaluation.csv`.

Repeat-last-week is the better forecast for Levi's SKUs. Across all four weeks its average error is 4.08, compared with 7.24 for Random Forest. At four weeks ahead the errors are 3.09 and 7.22. The baseline is closer on 718 of 896 scored weeks, and its typical-week error is 0 because many Levi's SKUs stay at the same demand from week to week.

Random Forest has only a modest advantage on The Souled Store data.

- Souled Store SKUs: average error is 5.59 for Random Forest and 5.97 for repeat-last-week. At four weeks ahead the errors are 6.37 and 7.03. Over all four weeks, repeat-last-week is still closer on more individual weeks (2,990 versus 2,762).
- Souled Store categories: average error is 12.63 for Random Forest and 13.43 for repeat-last-week. At four weeks ahead the errors are 14.97 and 15.50. The typical-week error over all four weeks is 2.00 for both methods.

Levi's categories are a small sample: 15 series-cutoff pairs. Random Forest has the lower average error there, 1,270.06 versus 1,742.63, mostly because of a few very large South Warehouse weeks. On a typical week, repeat-last-week is closer (median error 44.00 versus 174.52).

Use repeat-last-week for Levi's SKUs. Treat Random Forest as a small improvement for The Souled Store, not as a large gain.